<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/06-tablas-dinamicas-y-buscarv/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 0.6 · pandas II: tablas dinámicas, BUSCARV y estadística básica

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Calcular promedio, mediana y un resumen completo con una sola línea.
- Resumir datos con `groupby`, como una tabla dinámica rápida.
- Cruzar filas y columnas con `pivot_table`.
- Traer datos de otra tabla con `pd.merge()`, el `BUSCARV` (`VLOOKUP`) de Python.

## 📊 En Excel lo harías así

Te piden las ventas por categoría. Pero tu hoja de ventas solo trae el código del producto, así que:

1. Agregas una columna con `=BUSCARV()` (`VLOOKUP`) para traer la categoría desde la hoja de productos.
2. Arrastras la fórmula por 50 mil filas, y esperas.
3. Insertas una **tabla dinámica**.

En Python son los mismos pasos, y quedan escritos.

## 🐍 En Python

### Paso 1: abrir las tablas

Abrimos ventas y productos, ya limpios. Convertimos `fecha` y sacamos el año y el mes, como en el episodio 0.5. Este archivo trae todas las fechas en un solo formato.

In [1]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
ventas = pd.read_csv(DATOS + "ventas.csv")
productos = pd.read_csv(DATOS + "productos.csv")

ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="%Y-%m-%d")
ventas["anio"] = ventas["fecha"].dt.year
ventas["mes"] = ventas["fecha"].dt.month

ventas.head()

,id_venta,fecha,id_cliente,id_producto,id_vendedor,ciudad,cantidad,precio_unitario,descuento,total,metodo_pago,anio,mes
0,V000001,2024-01-02,C0002,P022,E006,Arequipa,3,48.0,0.0,144.0,Efectivo,2024,1
1,V000002,2024-01-02,C0003,P045,E004,Lima,2,46.0,0.0,92.0,Billetera digital,2024,1
2,V000003,2024-01-02,C0008,P021,E010,Chiclayo,3,46.0,0.0,138.0,Efectivo,2024,1
3,V000004,2024-01-02,C0022,P046,E007,Trujillo,9,70.0,0.0,630.0,Efectivo,2024,1
4,V000005,2024-01-02,C0024,P021,E013,Cusco,6,46.0,0.0,276.0,Crédito,2024,1


### Paso 2: estadística básica

`.mean()` es `=PROMEDIO()` (`AVERAGE`) y `.median()` es `=MEDIANA()` (`MEDIAN`):

In [3]:
print(round(ventas["total"].mean(), 2))
print(ventas["total"].median())

271.65
180.0


¿Por qué son tan distintos? La mediana es el valor del medio: la mitad de las ventas es de S/ 180 o menos. Las ventas grandes jalan el promedio: 1 de cada 10 concentra un tercio del monto. Si te piden "la venta típica", la mediana suele contar mejor la historia.

`.describe()` da todo el resumen de una vez:

In [4]:
ventas["total"].describe()

count    50000.000000
mean       271.645732
std        276.235954
min         16.500000
25%         92.000000
50%        180.000000
75%        351.000000
max       5440.500000
Name: total, dtype: float64

- `count`: cuántos datos hay, como `=CONTAR()` (`COUNT`) en una columna de números.
- `mean`, `min` y `max`: `=PROMEDIO()`, `=MIN()` y `=MAX()`.
- `std`: la desviación estándar, `=DESVEST.M()` (`STDEV.S`). Indica cuánto se alejan los datos del promedio.
- `25%`, `50%` y `75%`: los cuartiles, como `=CUARTIL()` (`QUARTILE`). El `50%` es la mediana.

### Paso 3: resumir con `groupby`

"Para cada ciudad, suma el total." Eso es `groupby`: elige por qué agrupar, qué columna mirar y qué cálculo hacer.

Como el resultado tiene una sola columna, `.sort_values()` no necesita un nombre:

In [5]:
ventas.groupby("ciudad")["total"].sum().sort_values(ascending=False)

ciudad
Lima        5203919.67
Trujillo    2069130.18
Arequipa    1967564.92
Cusco       1728419.55
Chiclayo    1441696.15
Piura       1171556.14
Name: total, dtype: float64

Se puede cambiar `.sum()` por `.mean()`, `.median()`, `.max()` o `.count()`.

### Paso 4: cruzar con `pivot_table`

`pivot_table` usa los mismos campos que arrastras en una tabla dinámica:

| En la tabla dinámica | En `pivot_table` |
|---|---|
| Filas | `index` |
| Columnas | `columns` |
| Valores | `values` |
| Sumar, promediar... | `aggfunc` |

`margins=True` agrega los totales, y `margins_name` les pone nombre. `.round()` quita los céntimos, aunque se sigue viendo un `.0`; con `.round(2)` dejarías dos decimales. Los totales redondeados pueden diferir en S/ 1; antes de redondear, cuadran.

In [6]:
ventas.pivot_table(
    index="ciudad",
    columns="anio",
    values="total",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
).round()

anio,2024,2025,Total
ciudad,,,
Arequipa,979282.0,988283.0,1967565.0
Chiclayo,705673.0,736024.0,1441696.0
Cusco,800147.0,928273.0,1728420.0
Lima,2588375.0,2615545.0,5203920.0
Piura,559419.0,612137.0,1171556.0
Trujillo,979149.0,1089982.0,2069130.0
Total,6612044.0,6970243.0,13582287.0


> ⚠️ Escribe siempre `aggfunc`. Si lo olvidas, `pivot_table` **promedia**, sin avisarte. Excel, en cambio, suma, salvo que la columna tenga vacíos o texto: entonces cuenta.

### Paso 5: `BUSCARV` con `pd.merge()`

`ventas` solo tiene el código del producto. La categoría está en `productos`. En Excel escribirías `=BUSCARV(código, tabla, columna, FALSO)` (`VLOOKUP`). Con `pd.merge()` indicas qué tablas cruzar y qué columna comparten, y siempre busca el código exacto:

- `on` es la columna que tienen en común.
- `how="left"` conserva todas las ventas, aunque no encuentre el código. Si no lo escribes, pandas usa `inner` y **borra** esas ventas sin avisar.
- De la otra tabla, elegimos las columnas que necesitamos, **siempre con la columna común**.

In [7]:
columnas = ["id_producto", "nombre", "categoria"]

ventas_completas = pd.merge(ventas, productos[columnas], on="id_producto", how="left")

ventas_completas.head()

,id_venta,fecha,id_cliente,id_producto,id_vendedor,ciudad,cantidad,precio_unitario,descuento,total,metodo_pago,anio,mes,nombre,categoria
0,V000001,2024-01-02,C0002,P022,E006,Arequipa,3,48.0,0.0,144.0,Efectivo,2024,1,Chicha morada 1 L,Bebidas
1,V000002,2024-01-02,C0003,P045,E004,Lima,2,46.0,0.0,92.0,Billetera digital,2024,1,Galletas rellenas de chocolate 6 unidades,Snacks
2,V000003,2024-01-02,C0008,P021,E010,Chiclayo,3,46.0,0.0,138.0,Efectivo,2024,1,Jugo de durazno 1 L,Bebidas
3,V000004,2024-01-02,C0022,P046,E007,Trujillo,9,70.0,0.0,630.0,Efectivo,2024,1,Papas fritas clásicas 150 g,Snacks
4,V000005,2024-01-02,C0024,P021,E013,Cusco,6,46.0,0.0,276.0,Crédito,2024,1,Jugo de durazno 1 L,Bebidas


Lo que `=BUSCARV()` (`VLOOKUP`) marca como `#N/D` (`#N/A`), aquí queda vacío. **Comprueba siempre** dos cosas: que sigan las mismas filas y que no haya vacíos. Importa porque `groupby` y `pivot_table` ignoran esas filas sin avisarte, y tu total saldría incompleto.

In [8]:
print(len(ventas), len(ventas_completas))
print(ventas_completas["categoria"].isna().sum())

50000 50000
0


> ⚠️ Si un código está repetido en la otra tabla, `=BUSCARV()` (`VLOOKUP`) toma el primero, pero `pd.merge()` **duplica la venta** y tus sumas salen infladas. Además, distingue mayúsculas: `p001` no es `P001`. Limpia los códigos primero, como en el 0.5.

Dos situaciones que te vas a encontrar:

- Si las columnas tienen **nombres distintos**, usa `left_on` y `right_on` en lugar de `on`. Por ejemplo, `id_vendedor` en `ventas` e `id_empleado` en `empleados`: `left_on="id_vendedor", right_on="id_empleado"`.
- Si las dos tablas tienen otra columna con el **mismo nombre**, como `ciudad`, pandas las renombra `ciudad_x` y `ciudad_y`. Se evita trayendo solo las columnas necesarias.

### Todo junto

Las ventas por categoría, el pedido del inicio:

In [9]:
ventas_completas.groupby("categoria")["total"].sum().sort_values(ascending=False)

categoria
Abarrotes           3884287.47
Cuidado personal    3303668.40
Limpieza            3043585.55
Bebidas             1868102.74
Snacks              1482642.45
Name: total, dtype: float64

Y ahora, una pregunta que ninguna hoja respondía sola: ¿en qué meses se vende el panetón?

In [10]:
paneton = ventas_completas[ventas_completas["nombre"] == "Panetón 900 g"]

paneton.groupby("mes")["total"].sum()

mes
1       4080.0
3       1080.0
4       1680.0
5       1974.0
6        600.0
7       5676.0
8       1800.0
9       1800.0
10     19800.0
11    158556.0
12    424734.0
Name: total, dtype: float64

Casi todo en noviembre y diciembre. Febrero no aparece porque no hubo ventas. Los datos confirman lo que cualquier bodeguero sabe, y esa es la idea: **primero preguntas, luego compruebas con los datos**.

## 🤖 Pídeselo a la IA

Dile cómo se llaman tus tablas y columnas, y qué resultado quieres. Usa nombres inventados, sin datos de tu empresa:

> *"Tengo un DataFrame `pedidos` con `id_tienda` y `monto`, y otro `tiendas` con `id_tienda` y `region`. Quiero el monto total por región. Muéstrame cómo hacerlo con `pd.merge()` y `groupby`, y explícame cada parte."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla y comprueba que los totales cuadren.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| `=PROMEDIO()` (`AVERAGE`) | `.mean()` |
| `=MEDIANA()` (`MEDIAN`) | `.median()` |
| Resumen de una columna | `.describe()` |
| Tabla dinámica de una dimensión | `ventas.groupby("ciudad")["total"].sum()` |
| Tabla dinámica con filas y columnas | `ventas.pivot_table(index=..., columns=..., values=..., aggfunc="sum")` |
| Totales generales | `margins=True` |
| `=BUSCARV()` (`VLOOKUP`) | `pd.merge(ventas, productos, on="id_producto", how="left")` |
| `=BUSCARV()` con columnas de nombres distintos | `left_on="id_vendedor", right_on="id_empleado"` |
| `#N/D` (`#N/A`) | Celda vacía: cuéntalas con `.isna().sum()` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/06-tablas-dinamicas-y-buscarv/ejercicio.ipynb) y responde las preguntas de la gerencia. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/06-tablas-dinamicas-y-buscarv/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0